In [8]:
# Files
database_file = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/BMI/bmi_database.fasta"
primer_table = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/BMI/bmi-primers.tsv"
run_name = 'bmi-phyl'
output_folder = '/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data' + '/' + run_name
otl_folder = "/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized/bmi/filtered_phyl"

In [2]:
# Intake data
from src.reference_database.sequence_import import *

custom_fasta_import = CustomFastaImport(database_file)
custom_fasta_import.read_fasta(database_file, check_taxid=False)
data = custom_fasta_import.data
data.head()

,seq_id,sequence,length,taxa_info
0,AASFB602-10,AACTTTATATTTAATATTTGGTGTATGATCTGCTATAGCGGGAACA...,658,nan|Ctenidae|family|Animalia|Arthropoda|Arachn...
1,AASFB633-10,TACTTTATATTTAATTTTTGGAGCTTGATCTGCTATAGTTGGAACA...,658,nan|Clubionidae|family|Animalia|Arthropoda|Ara...
2,AASFB645-10,TTCTTTGTATTTATTATTTGGTAGATGATCTGCTATAGTAGGAACA...,658,nan|Anyphaenidae|family|Animalia|Arthropoda|Ar...
3,AASFB649-10,AACTTTATATTTGGTATTTGGGGCTTGATCAGCTATAGTTGGAATA...,658,nan|Corinnidae|family|Animalia|Arthropoda|Arac...
4,AASFB654-10,AACACTTTATTTAGTTTTTGGAGCTTGATCAGCTATAGTTGGTACT...,658,nan|Salticidae|family|Animalia|Arthropoda|Arac...


In [3]:
processed_data = custom_fasta_import.pre_process_harmonized_fasta_database()
processed_data.head()

mozaiko INFO: This method assumes that the FASTA header has been harmonized andcontains the following pipe-separated information: 
sequence id, original taxonomy, harmonized taxonomy, rank, kingdom, phylum, class, order, family, genus.


,seq_id,sequence,length,all_taxa_info,original_taxa_info,scientificName,rank,kingdom,phylum,class,order,family,genus,species
0,AASFB602-10,AACTTTATATTTAATATTTGGTGTATGATCTGCTATAGCGGGAACA...,658,nan|Ctenidae|family|Animalia|Arthropoda|Arachn...,nan,Ctenidae,family,Animalia,Arthropoda,Arachnida,Araneae,Ctenidae,NaN,NaN
1,AASFB633-10,TACTTTATATTTAATTTTTGGAGCTTGATCTGCTATAGTTGGAACA...,658,nan|Clubionidae|family|Animalia|Arthropoda|Ara...,nan,Clubionidae,family,Animalia,Arthropoda,Arachnida,Araneae,Clubionidae,NaN,NaN
2,AASFB645-10,TTCTTTGTATTTATTATTTGGTAGATGATCTGCTATAGTAGGAACA...,658,nan|Anyphaenidae|family|Animalia|Arthropoda|Ar...,nan,Anyphaenidae,family,Animalia,Arthropoda,Arachnida,Araneae,Anyphaenidae,NaN,NaN
3,AASFB649-10,AACTTTATATTTGGTATTTGGGGCTTGATCAGCTATAGTTGGAATA...,658,nan|Corinnidae|family|Animalia|Arthropoda|Arac...,nan,Corinnidae,family,Animalia,Arthropoda,Arachnida,Araneae,Corinnidae,NaN,NaN
4,AASFB654-10,AACACTTTATTTAGTTTTTGGAGCTTGATCAGCTATAGTTGGTACT...,658,nan|Salticidae|family|Animalia|Arthropoda|Arac...,nan,Salticidae,family,Animalia,Arthropoda,Arachnida,Araneae,Salticidae,NaN,NaN


In [4]:
processed_data[processed_data['rank'] == 'order']

,seq_id,sequence,length,all_taxa_info,original_taxa_info,scientificName,rank,kingdom,phylum,class,order,family,genus,species


In [5]:
database_processed_path = custom_fasta_import.database_fasta_file

In [6]:
from src.in_silico_analysis.amplification import InSilicoAmplification

insil = InSilicoAmplification(database_processed_path, run_name=run_name)
insil.run_in_silico_analysis(primer_table)

mozaiko INFO: Checking if cutadapt is installed...
mozaiko INFO: cutadapt is installed.
mozaiko INFO: Checking if CRABS is installed...
mozaiko INFO: CRABS is installed.
mozaiko INFO: To continue the analysis, a set of primers is needed. This information should be uploaded as a TSV table and it should contain the following fields: ['target_group', 'barcode_region', 'assay_name', 'fwd_seq', 'rev_seq']
mozaiko INFO: All set. Running in-silico amplification...
mozaiko INFO: Running cutadapt command as: cutadapt -g GGDACWGGWTGAACWGTWTAYCCHCC;min_overlap=26...CCWGTWYTAGCHGGDGCWATYAC;min_overlap=23 --output data/output_data/bmi-phyl/amplicon/COI_FwhF2_FwhR2n.fasta /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/BMI/bmi_database_processed.fasta --no-indels -e 3 --revcomp --quiet --minimum-length 50 --maximum-length 551 --discard-untrimmed --action retain
mozaiko INFO: Running cutadapt command as: cutadapt -g GGDACWGGWTGAACWGTWTAYCCHCC;min_overlap=26...CCWGTWY

In [10]:
from src.marker_scoring.metrics_system import *

for otl in os.listdir(otl_folder):
    otl_path = os.path.join(otl_folder, otl)
    country_name = otl.split('_')[0]
    output_path = os.path.join(output_folder, country_name + '_ranked_primers.tsv')
    print(" ---------------------")
    print("Processing OTL:", otl)
    mex = MetricsSystemExecutor(results_folder=output_folder, otl=otl_path, primer_table=primer_table)
    binding_dataframe = mex.comprehensive_primer_analysis(mex.results_folder, save_otl_level_results=True)
    binding_dataframe.to_csv(output_path, sep="\t", index=False)

 ---------------------
Processing OTL: Sweden_taxonlist_bmi_harmonized.tsv
Set insert_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/insert/filtered, amplicon_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/amplicon and incomplete_pbs_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/incomplete_pbs/filtered/filtered_intersection.
mozaiko INFO: Retrieving primer-PBS statistics.
mozaiko WARNING: Empty data in amplicon or insert file for COI_JgLCO1490_jgHCO2198.fasta
mozaiko WARNING: Missing data for COI_JgLCO1490_jgHCO2198, skipping...
mozaiko INFO: Retriving information on the Reference Database Quality.
mozaiko INFO: Calculating the number of barcodes per FASTA.
mozaiko INFO: Calculating the number of barcodes per FASTA.
 ---------------------
Processing OTL: Austria_taxonlist_bmi_harmonized.tsv
Set insert_folder_path t

In [9]:
# Create Multibarcode Input
from src.marker_scoring.metrics_system import *

input_path_mb = output_folder + '/bmi_multibarcode_input.tsv'
create_multibarcode_input_for_bmi(output_folder, input_path_mb)

mozaiko INFO: MultiBarcodeTools file created to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/bmi_multibarcode_input.tsv
mozaiko INFO: Total entries processed: 3702187
mozaiko WARNING: 0 entries were not written because they did not meet the minimum length of columns. Check if headers have been through harmonization.
mozaiko WARNING: 0 entries were not written because of unexpected header format.


In [3]:
# Remove 'rc' from FASTA headers
from src.marker_scoring.scoring_utils import remove_rc_suffix_from_fasta_files
remove_rc_suffix_from_fasta_files(output_folder)

{'files_processed': 82, 'headers_modified': 1332, 'errors': []}

In [ ]:
# Populate 'Species' columns with family detailed in 'SourceFolder' when 'Species' is NaN"

import pandas as pd
import os

output_folder = '/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/multibarcode'

for file in os.listdir(output_folder):
    if file.endswith('.xlsx'):
        xlsx_path = os.path.join(output_folder, file)
        try:
            df = pd.read_excel(xlsx_path, engine='openpyxl')
            
            if 'Species' in df.columns and 'SourceFolder' in df.columns:
                df['Species'] = df['Species'].fillna(df['SourceFolder'])
                df = df.drop(columns=['SourceFolder'])
                
                tsv_path = xlsx_path.replace('.xlsx', '.tsv')
                df.to_csv(tsv_path, sep="\t", index=False)
                print(f"Processed {file}")
            else:
                print(f"Required columns not found in {file}")
        except Exception as e:
            print(f"Error processing {file}: {e}")

Processed merged_matrices_COI_FwhF2_FwhR2n.xlsx
Processed merged_matrices_COI_FwhF2_EPTDr2n.xlsx
Processed merged_matrices_COI_mICOIintF_jgHCO2198.xlsx
Processed merged_matrices_COI_MICOIntF_HCO2198.xlsx
Processed merged_matrices_COI_BF2_BR2.xlsx
Processed merged_matrices_COI_fwhF2_Fol-degen-rev.xlsx
Processed merged_matrices_COI_MG2-LCO1490_MG2-univR.xlsx
Processed merged_matrices_COI_BF3_BR2.xlsx


In [2]:
# Combine all results
import pandas as pd
import os
import glob


tsv_files = glob.glob(os.path.join(output_folder, '*.tsv'))

merged_df = pd.DataFrame()
merged_df['Species'] = []

for tsv_file in tsv_files:
    df = pd.read_csv(tsv_file, sep='\t')
    if 'Species' not in df.columns:
        print(f"Warning: File {tsv_file} doesn't have a 'Species' column. Skipping.")
        continue
    
    if len(df.columns) < 2:
        print(f"Warning: File {tsv_file} only has Species column, no primer data. Skipping.")
        continue

    primer_column = df.columns[1]
    
    # Create a simplified dataframe with just Species and the primer column
    species_df = df[['Species', primer_column]]
    
    if merged_df.empty:
        merged_df = species_df
    else:
        merged_df = pd.merge(merged_df, species_df, on='Species', how='outer')

merged_df = merged_df.sort_values('Species')

if not merged_df.empty:
    output_path = os.path.join(output_folder, 'all_primers_merged.tsv')
    merged_df.to_csv(output_path, sep='\t', index=False)
    
    print(f"Successfully merged {len(tsv_files)} TSV files into {output_path}")
    print(f"Total species in merged file: {len(merged_df)}")
    print(f"Columns in final file: {', '.join(merged_df.columns)}")
else:
    print("No data to merge.")


Successfully merged 9 TSV files into /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/multibarcode/all_primers_merged.tsv
Total species in merged file: 9491588
Columns in final file: Species, COI_FwhF2_FwhR2n, COI_FwhF2_EPTDr2n, COI_mICOIintF_jgHCO2198, COI_fwhF2_Fol_degen_rev, COI_BF2_BR2, COI_BF3_BR2, COI_MG2_LCO1490_MG2_univR, COI_MICOIntF_HCO2198


In [6]:
# Count total unique number of species in all tsv folders  with 'merged_matrices' in name

tsv_files = glob.glob(os.path.join(output_folder, '*.tsv'))

#check if file has 'merged_matrices' in name
tsv_files = [file for file in tsv_files if 'merged_matrices' in file]

# check all species, not only unique
species = []
for tsv_file in tsv_files:
    df = pd.read_csv(tsv_file, sep='\t')
    species.extend(df['Species'].tolist())

In [7]:
len(species)

282465

In [4]:
from src.marker_scoring.metrics_system import *

otl_path = '/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized/bmi/filtered_phyl/Austria_taxonlist_bmi_harmonized.tsv'
country_name = otl_path.split('/')[-1]
country_name = country_name.split('_')[0]
output_path = os.path.join(output_folder, country_name + '_ranked_primers_v2_merged.tsv')
mex = MetricsSystemExecutor(results_folder=output_folder, otl=otl_path, primer_table=primer_table)
traits_res_df = mex.get_traits_and_resolution(run_multibarcode_pipeline=False)

Set insert_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/insert/filtered, amplicon_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/amplicon and incomplete_pbs_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/incomplete_pbs/filtered/filtered_intersection.
results_folder: /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl, insert_folder_path: None, amplicon_folder_path: None
Set insert_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/insert/filtered, amplicon_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/amplicon and incomplete_pbs_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/incomplete_pbs/filtered/filtered_

In [5]:
traits_res_df

,taxonomic_resolution
primer,
COI_BF3_BR2,1.29


In [ ]:
/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/bmi-phyl/mulitbarcode